 ***Olist E-commerce financial analysis project PART 2*** -



**BASED ON THE MY WORK(RECORD TO REPORT)**

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)


orders = pd.read_csv('olist_orders_dataset.csv')
payments = pd.read_csv('olist_order_payments_dataset.csv')
order_items = pd.read_csv('olist_order_items_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')

df = pd.merge(orders, payments, on='order_id', how='left')
df = pd.merge(df, order_items, on='order_id', how='left')
df = pd.merge(df, customers, on='customer_id', how='left')

In [8]:
# Calculate Order Value(expected Amount)


order_value = df.groupby('order_id').agg({
    'price': 'sum',
    'freight_value': 'sum'
}).reset_index()

order_value['expected_value'] = order_value['price'] + order_value['freight_value']


In [12]:
# Calculate Payment Recieved

payment_value = df.groupby('order_id').agg({
    'payment_value': 'sum'
}).reset_index()

In [18]:
# Comparing expected value vs the actual payment received

recon = pd.merge(order_value, payment_value, on='order_id', how='inner')

recon['difference'] = (recon['payment_value'] - recon['expected_value']).round(2)

In [19]:
# identify Issues

recon_issues = recon[recon['difference'] != 0]

recon_issues.head()

,order_id,price,freight_value,expected_value,payment_value,difference
13,0008288aa423d2a3f00fcb17cd7d8719,99.80,26.74,126.54,253.08,126.54
24,0010dedd556712d7bb69a19cb7bbd37a,0.00,0.00,0.00,111.12,111.12
32,00143d0f86d6fbd9f9b38ab440ac16f5,63.99,45.30,109.29,327.87,218.58
36,0016dfedd97fc2950e388d2971d718c7,99.50,41.60,141.10,70.55,-70.55
40,001ab0a7578dd66cd4b0a71f5b6e1e41,74.67,52.89,127.56,382.68,255.12


In [23]:
#Summary of issues
(recon_issues['difference'].describe()).round(2)

,difference
count,13273.00
mean,294.13
std,1383.37
min,-12823.72
25%,53.42
50%,141.86
75%,294.71
max,95648.56


In [24]:
# Checking for the cancelled orders

cancelled_orders = df[df['order_status'] == 'canceled']
len(cancelled_orders)

745

In [25]:
# Checking for delivery delays

df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'])
df['order_estimated_delivery_date'] = pd.to_datetime(df['order_estimated_delivery_date'])

delayed_orders = df[df['order_delivered_customer_date'] > df['order_estimated_delivery_date']]
len(delayed_orders)

9027

In [26]:
# checking for high freight cost impact

df['freight_ratio'] = df['freight_value'] / df['price']

high_freight = df[df['freight_ratio'] > 0.5]
high_freight.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,payment_sequential,payment_type,payment_installments,payment_value,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,freight_ratio
5,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,1.0,credit_card,1.0,72.20,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,0.604444
10,76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06,1.0,boleto,1.0,35.95,1.0,ac1789e492dcd698c5c10b97a671243a,63b9ae557efed31d1f7687917d248a8d,2017-01-27 18:29:09,19.90,16.05,39382392765b6dc74812866ee5ee92a7,99655,faxinalzinho,RS,0.806533
16,82566a660a982b15fb86e904c8d32918,d3e3b74c766bc6214e0c830b17ee2341,delivered,2018-06-07 10:06:19,2018-06-09 03:13:12,2018-06-11 13:29:00,2018-06-19 12:05:52,2018-07-18,1.0,boleto,1.0,50.13,1.0,72a97c271b2e429974398f46b93ae530,094ced053e257ae8cae57205592d6712,2018-06-18 03:13:12,31.90,18.23,e97109680b052ee858d93a539597bba7,35400,ouro preto,MG,0.571473
17,5ff96c15d0b717ac6ad1f3d77225a350,19402a48fe860416adf93348aba37740,delivered,2018-07-25 17:44:10,2018-07-25 17:55:14,2018-07-26 13:16:00,2018-07-30 15:52:25,2018-08-08,1.0,credit_card,3.0,32.70,1.0,10adb53d8faa890ca7c2f0cbcb68d777,1900267e848ceeba8fa32d80c1a5f5a8,2018-07-27 17:55:14,19.90,12.80,e2dfa3127fedbbca9707b36304996dab,4812,sao paulo,SP,0.643216
21,116f0b09343b49556bbad5f35bee0cdf,3187789bec990987628d7a9beb4dd6ac,delivered,2017-12-26 23:41:31,2017-12-26 23:50:22,2017-12-28 18:33:05,2018-01-08 22:36:36,2018-01-29,1.0,credit_card,4.0,43.09,1.0,a47295965bd091207681b541b26e40a5,ea8482cd71df3c1969d7b9473ff13abc,2018-01-02 23:50:22,27.99,15.10,6087cfc70fd833cf2db637a5e6e9d76b,88780,imbituba,SC,0.539478


In [27]:
# cheking for Paymmet method that dominates

payment_analysis = df.groupby('payment_type')['payment_value'].sum().reset_index()
payment_analysis

,payment_type,payment_value
0,boleto,4086820.71
1,credit_card,15694885.84
2,debit_card,256417.92
3,not_defined,0.00
4,voucher,432602.19
